# z719 final gerencial v3: feature engineering
Francisco Medina Raed. Sigue al WF7191 (V3 desde 202103, grid por ganancia en 202107, semillerío de 20) y agrega **cuatro cambios de feature engineering que se prenden o apagan por separado** en la primera celda.

| Bloque | Parámetro | Qué hace |
|---|---|---|
| Sacar IDs | `PARAM$fe$sacar_ids` | `numero_de_cliente`, `foto_mes` y `kmes` (con sus lags y deltas) dejan de ser variables del modelo. Siguen en el dataset para los lags, los filtros y el archivo de Kaggle. |
| 1. Rankings | `PARAM$fe$rankings` | Data drifting: cada monto (`m...`, `Visa_m...`, `Master_m...`) se reemplaza por su posición relativa dentro del mes, con el cero fijo: positivos en (0, 1], negativos en [−1, 0), el cero queda en 0. Se hace antes de los lags, así los deltas también quedan sin inflación. |
| 2. Ratios | `PARAM$fe$ratios` | Hasta 10 variables nuevas dentro del mes: consumo y transacciones totales de tarjetas, uso del límite de Visa y de Master, tarjeta en cierre, proporción de operatoria por homebanking, cobra el sueldo en el banco, saldo y consumo medidos en sueldos, transacciones por producto. Las que necesitan una columna que no está en el dataset se saltean y se avisa. |
| 3. Tendencias | `PARAM$fe$tendencias` | Para las 40 variables más importantes del WF7191: pendiente, promedio, máximo, mínimo y valor actual sobre el promedio, en una ventana de 6 meses. |

**Cómo comparar los bloques:** cada combinación de bloques tiene su propio nombre (por ejemplo `V3fe_noid_rank_ratio_tend` o `V3fe_rank`), su propia carpeta dentro de `WF7193` y sus propios archivos de Kaggle. Se puede correr con todo prendido, o cambiar los parámetros y volver a correr todas las celdas: no se pisan. La penúltima celda compara todas las corridas por su **ganancia en julio**, sin gastar envíos; la última muestra las curvas del Public junto a las del WF7190 y el WF7191.

**Requisitos**
- El bloque de tendencias necesita el `impo.txt` del WF7191 (`WF7191/V3gan_desde202103/impo.txt`) para elegir las 40 variables. La lista queda guardada en `tendencias_cols.txt`.
- Tiempo: el grid son 90 entrenamientos, como en el WF7191, pero con más columnas. Primero conviene `modo <- "test"`.

**Cambios de esta versión (experimento 7193)**
- **El grid evalúa cada combinación con 10 semillas** (antes 3).
- **No repite combinaciones equivalentes.** Un árbol no puede tener más hojas que filas / `min_data_in_leaf`; por encima de ese tope, cambiar `num_leaves` da el mismo modelo. De las 30 combinaciones quedan 16 distintas.
- **Control de configuración.** Cada carpeta guarda con qué configuración se hicieron sus resultados (`config_corrida.yml`). Si se cambia un parámetro y se vuelve a correr sobre la misma carpeta, el notebook se detiene en lugar de reutilizar resultados viejos. Para cambiar la configuración, cambiar `PARAM$experimento`.
- Usa la misma lista de variables para tendencias que el WF7192.


## Parámetros, paquetes y carpetas

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

modo <- "full"           # "test" = corrida corta para probar que todo anda (no sube a Kaggle)
subir_kaggle <- TRUE     # sube los archivos a Kaggle apenas se generan (necesita el kaggle.json)

PARAM <- list()
PARAM$semilla_primigenia <- 123457     # <- tu semilla primigenia
PARAM$experimento <- 7193
PARAM$comparar_con <- c(7190, 7191, 7192)           # experimentos anteriores que se muestran en la curva de puntajes
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# --- Feature engineering: cada bloque se prende o se apaga por separado ---
# Cada combinacion de bloques se guarda en su propia carpeta y con su propio nombre en Kaggle,
# asi se pueden correr de a uno y comparar sin pisarse.
PARAM$fe$sacar_ids  <- TRUE     # numero_de_cliente, foto_mes y kmes dejan de ser variables del modelo
PARAM$fe$ratios     <- TRUE     # bloque 2: ratios y sumas dentro del mes (senales de fuga)
PARAM$fe$rankings   <- TRUE     # bloque 1: montos convertidos a ranking por mes (data drifting)
PARAM$fe$tendencias <- TRUE     # bloque 3: tendencia, promedio, max, min y ratio en ventana, sobre las top variables
PARAM$fe$tendencias_top     <- 40
PARAM$fe$tendencias_ventana <- 6

etiqueta <- paste(c(if (PARAM$fe$sacar_ids) "noid", if (PARAM$fe$rankings) "rank",
                    if (PARAM$fe$ratios) "ratio", if (PARAM$fe$tendencias) "tend"), collapse = "_")
if (etiqueta == "") etiqueta <- "base"

# --- Training strategy: la ganadora del Exp 07, del WF7190 y del WF7191 ---
PARAM$variantes <- list(
  list(nombre = paste0("V3fe_", etiqueta), desde = 202103,
       descripcion = paste0("training 202103-202106, final_train 202103-202107, grid por ganancia, FE: ", etiqueta))
)
PARAM$trainingstrategy$validate     <- 202107
PARAM$trainingstrategy$hasta        <- 202106   # ultimo mes de training
PARAM$trainingstrategy$future       <- 202109
PARAM$trainingstrategy$training_pct <- 1.0      # sin undersampling, como el z719
PARAM$trainingstrategy$positivos    <- c("BAJA+1", "BAJA+2")

# --- Ganancia de la competencia ---
PARAM$ganancia$acierto <- 975000    # por cada BAJA+2 contactado
PARAM$ganancia$error   <- -25000    # por cada BAJA+1 o CONTINUA contactado
PARAM$ganancia$ventana <- 401       # suavizado de la curva de ganancia en validacion (clientes)

# --- Grid search: ahora se elige por GANANCIA en 202107, no por AUC ---
# min_data_in_leaf arranca en 64: con 16 y 32 el WF7190 sobreajusto (V2 se desplomo en Kaggle)
PARAM$grid <- list(
  num_leaves       = c(64, 128, 256, 512, 1024),
  min_data_in_leaf = c(64, 128, 256, 512, 1024, 2048)
)
PARAM$grid$semillas_eval <- 10      # cada combinacion se evalua con 10 semillas y se promedia (la ganancia es ruidosa)

# --- Parametros fijos de LightGBM: identicos al z719 ---
PARAM$lgbm$param_fijos <- list(
  objective = "binary",
  metric = "None",              # la metrica es la ganancia (feval), no el AUC
  first_metric_only = TRUE,
  boost_from_average = TRUE,
  feature_pre_filter = FALSE,
  verbosity = -100,
  force_row_wise = TRUE,
  seed = PARAM$semilla_primigenia,
  max_bin = 31,
  learning_rate = 0.03,
  feature_fraction = 0.5,
  num_iterations = 2048,        # lo limita el early stopping
  early_stopping_rounds = 200,
  num_leaves = 64,
  min_data_in_leaf = 128
)

# --- Semillerio: el modelo final se entrena con N semillas y se promedian las probabilidades ---
PARAM$semillerio$cantidad <- 20

# --- Opcional: escalar num_iterations por las filas que agrega 202107 al final_train ---
PARAM$final$escalar_iteraciones <- FALSE

# --- Kaggle: cortes de a 50 alrededor del pico del Exp 07 (unos 1.100 envios) ---
PARAM$kaggle$competencia <- "labo-1-virtual-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1400, by = 50)

if (modo == "test") {
  PARAM$experimento <- paste0(PARAM$experimento, "_test")
  PARAM$grid <- list(num_leaves = c(64, 256), min_data_in_leaf = c(64, 512), semillas_eval = 2)
  PARAM$lgbm$param_fijos$num_iterations <- 60
  PARAM$lgbm$param_fijos$early_stopping_rounds <- 20
  PARAM$semillerio$cantidad <- 3
  subir_kaggle <- FALSE
}

# semillas del semillerio: la primera es la primigenia, el resto se derivan de ella (reproducible)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$semillerio$semillas <- c(PARAM$semilla_primigenia,
  setdiff(sample(100000:999999, PARAM$semillerio$cantidad + 5), PARAM$semilla_primigenia)[seq_len(PARAM$semillerio$cantidad - 1)])

for (paquete in c("data.table", "R.utils", "lightgbm", "yaml")) {
  if (!require(paquete, character.only = TRUE)) {
    install.packages(paquete)
    require(paquete, character.only = TRUE)
  }
}

# carpeta del experimento en el bucket de la VM (o en el home si no hay bucket)
base <- path.expand(if (dir.exists("~/buckets/b1")) "~/buckets/b1/exp" else "~/exp")
carpeta_exp <- file.path(base, paste0("WF", PARAM$experimento))
dir.create(carpeta_exp, recursive = TRUE, showWarnings = FALSE)
setwd(carpeta_exp)

# dataset: lo busco donde lo suele dejar la VM; si no esta, lo descargo
candidatos <- path.expand(file.path(c("~/datasets", "~/buckets/b1/datasets", carpeta_exp), PARAM$dataset))
archivo_dataset <- candidatos[file.exists(candidatos)][1]
if (is.na(archivo_dataset)) {
  archivo_dataset <- file.path(carpeta_exp, PARAM$dataset)
  download.file(paste0("https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/", PARAM$dataset),
                archivo_dataset, mode = "wb")
}

message("variante: ", PARAM$variantes[[1]]$nombre, "\ncarpeta: ", carpeta_exp, "\ndataset: ", archivo_dataset,
        "\nsemillas del semillerio: ", paste(PARAM$semillerio$semillas, collapse = ", "))

## Preprocesamiento con los bloques de feature engineering
Orden: catastrophe analysis → FE intra-mes del z719 → **ratios** (sobre montos nominales del mismo mes) → **rankings por mes** → lags y deltas del z719 → **tendencias** → se definen las variables del modelo (**sin IDs**) → los Inf y NaN de las divisiones por cero pasan a NA.

In [ ]:
dataset <- fread(archivo_dataset)

# ---------- 6.3.1.2 Catastrophe Analysis (identico al z719) ----------
dataset[foto_mes == 202006, internet := NA]
dataset[foto_mes == 202006, mrentabilidad := NA]
dataset[foto_mes == 202006, mrentabilidad_annual := NA]
dataset[foto_mes == 202006, mcomisiones := NA]
dataset[foto_mes == 202006, mactivos_margen := NA]
dataset[foto_mes == 202006, mpasivos_margen := NA]
dataset[foto_mes == 202006, mcuentas_saldo := NA]
dataset[foto_mes == 202006, ctarjeta_visa_transacciones := NA]
dataset[foto_mes == 202006, mtarjeta_visa_consumo := NA]
dataset[foto_mes == 202006, mtarjeta_master_consumo := NA]
dataset[foto_mes == 202006, ccallcenter_transacciones := NA]
dataset[foto_mes == 202006, chomebanking_transacciones := NA]

# ---------- 6.3.1.3 FE intra-mes (identico al z719) ----------
atributos_presentes <- function(patributos) {
  atributos <- unique(patributos)
  length(atributos) == length(intersect(atributos, colnames(dataset)))
}
if (atributos_presentes(c("foto_mes")))
  dataset[, kmes := foto_mes %% 100]
if (atributos_presentes(c("mpayroll", "cliente_edad")))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# ---------- BLOQUE 2: ratios y sumas dentro del mes ----------
# Se calculan ANTES de los rankings: un cociente entre dos montos del mismo mes no tiene inflacion.
# Las sumas de montos se llaman m... (despues se rankean); los cocientes se llaman r_... (no se rankean).
fe_creadas <- character(); fe_salteadas <- character()
if (PARAM$fe$ratios) {
  # suma por fila que da NA solo si todos los sumandos son NA (no revive los ceros de 202006)
  suma_na <- function(...) {
    m <- cbind(...)
    s <- rowSums(m, na.rm = TRUE)
    s[rowSums(!is.na(m)) == 0] <- NA
    s
  }
  # crea la variable solo si estan todas las columnas que necesita; si no, la saltea y avisa
  agregar <- function(nombre, requeridos, expr) {
    if (atributos_presentes(requeridos)) {
      dataset[, (nombre) := eval(expr)]
      fe_creadas <<- c(fe_creadas, nombre)
    } else {
      fe_salteadas <<- c(fe_salteadas, paste0(nombre, " (falta: ",
        paste(setdiff(requeridos, colnames(dataset)), collapse = ", "), ")"))
    }
  }

  # actividad total con tarjetas de credito
  agregar("mtarjetas_consumo", c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"),
          quote(suma_na(mtarjeta_visa_consumo, mtarjeta_master_consumo)))
  agregar("ctarjetas_transacciones", c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones"),
          quote(suma_na(ctarjeta_visa_transacciones, ctarjeta_master_transacciones)))
  # cuanto del limite de la tarjeta esta usando
  agregar("r_visa_uso_limite", c("Visa_msaldototal", "Visa_mlimitecompra"),
          quote(Visa_msaldototal / Visa_mlimitecompra))
  agregar("r_master_uso_limite", c("Master_msaldototal", "Master_mlimitecompra"),
          quote(Master_msaldototal / Master_mlimitecompra))
  # alguna tarjeta en proceso de cierre o cerrada (status 6, 7 o 9)
  agregar("tarjeta_cerrando", c("Visa_status", "Master_status"),
          quote(as.integer(Visa_status %in% c(6, 7, 9) | Master_status %in% c(6, 7, 9))))
  # que parte de la operatoria es por homebanking
  agregar("r_digital", c("chomebanking_transacciones", "ccallcenter_transacciones", "ccajas_transacciones"),
          quote(chomebanking_transacciones /
                (chomebanking_transacciones + ccallcenter_transacciones + ccajas_transacciones)))
  # cobra el sueldo en el banco; su delta1 = -1 es "dejo de cobrar el sueldo aca"
  agregar("tiene_payroll", c("mpayroll"), quote(as.integer(mpayroll > 0)))
  # saldo y consumo medidos en sueldos
  agregar("r_saldo_sobre_payroll", c("mcuentas_saldo", "mpayroll"),
          quote(ifelse(mpayroll > 0, mcuentas_saldo / mpayroll, NA_real_)))
  agregar("r_consumo_sobre_payroll", c("mtarjetas_consumo", "mpayroll"),
          quote(ifelse(mpayroll > 0, mtarjetas_consumo / mpayroll, NA_real_)))
  # intensidad de uso: transacciones del trimestre por producto contratado
  agregar("r_trx_por_producto", c("ctrx_quarter", "cproductos"), quote(ctrx_quarter / cproductos))

  message("Bloque ratios: creadas ", length(fe_creadas), ": ", paste(fe_creadas, collapse = ", "))
  if (length(fe_salteadas)) message("  salteadas ", length(fe_salteadas), ": ", paste(fe_salteadas, collapse = "; "))
}

# ---------- BLOQUE 1: data drifting, rankings por mes de los montos ----------
# Cada monto se reemplaza por su posicion relativa dentro del mes, con el cero fijo:
# positivos en (0, 1], negativos en [-1, 0), el cero queda en 0 y los NA quedan NA.
cols_monetarias <- character()
if (PARAM$fe$rankings) {
  rank_cero_fijo <- function(x) {
    r <- rep(NA_real_, length(x))
    pos <- which(x > 0); neg <- which(x < 0)
    r[which(x == 0)] <- 0
    if (length(pos)) r[pos] <- frank(x[pos], ties.method = "average") / length(pos)
    if (length(neg)) r[neg] <- -frank(-x[neg], ties.method = "average") / length(neg)
    r
  }
  cols_monetarias <- grep("^(m|Visa_m|Master_m)", colnames(dataset), value = TRUE)
  # paso todo a numerico antes: una columna entera truncaria los rankings a 0
  for (v in cols_monetarias) set(dataset, j = v, value = as.numeric(dataset[[v]]))
  dataset[, (cols_monetarias) := lapply(.SD, rank_cero_fijo), by = foto_mes, .SDcols = cols_monetarias]
  message("Bloque rankings: ", length(cols_monetarias), " variables de monto convertidas a ranking por mes")
}

# ---------- 6.3.1.5 FE historico: lags y deltas de orden 1 y 2 (identico al z719) ----------
# los lags usan la fila anterior de cada cliente: verifico que esten en orden cronologico
stopifnot(dataset[, all(diff(foto_mes) > 0), by = numero_de_cliente][, all(V1)])

cols_lagueables <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
dataset[, paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
dataset[, paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
        by = numero_de_cliente, .SDcols = cols_lagueables]
for (vcol in cols_lagueables) {
  dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
  dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# ---------- BLOQUE 3: tendencias en ventana, sobre las variables mas importantes ----------
cols_top <- character()
if (PARAM$fe$tendencias) {
  n <- PARAM$fe$tendencias_ventana

  # lista de variables: se fija una sola vez y queda guardada en la carpeta del experimento (reproducible)
  archivo_cols <- file.path(carpeta_exp, "tendencias_cols.txt")
  if (file.exists(archivo_cols)) {
    cols_top <- readLines(archivo_cols)
  } else {
    # si otro experimento ya fijo la lista (por ejemplo el WF7192), uso exactamente la misma
    previas <- file.path(base, paste0("WF", PARAM$comparar_con), "tendencias_cols.txt")
    previas <- previas[file.exists(previas)]
    candidatos <- file.path(base, c("WF7191/V3gan_desde202103", "WF7191_test/V3gan_desde202103",
                                    "WF7190/V3_desde202103"), "impo.txt")
    origen <- candidatos[file.exists(candidatos)][1]
    if (length(previas)) {
      file.copy(previas[length(previas)], archivo_cols)
      message("Variables para tendencias copiadas de ", previas[length(previas)])
      cols_top <- readLines(archivo_cols)
    } else {
    if (is.na(origen))
      stop("No encuentro el impo.txt del WF7191 ni del WF7190 para elegir las variables de las tendencias. ",
           "Copialo a una de esas carpetas o poné PARAM$fe$tendencias <- FALSE")
    impo <- fread(origen)                                   # viene ordenado por importancia (Gain)
    base_vars <- unique(sub("_(lag|delta)[12]$", "", impo$Feature))
    cols_top <- head(intersect(base_vars, setdiff(cols_lagueables, "kmes")), PARAM$fe$tendencias_top)
    writeLines(cols_top, archivo_cols)
    message("Variables para tendencias tomadas de ", origen)
    }
  }
  cols_top <- intersect(cols_top, cols_lagueables)
  if (!length(cols_top)) stop("La lista de variables para tendencias quedo vacia: revisar ", archivo_cols)

  # trabajo sobre una copia ordenada por cliente y mes, sin cambiar el orden de las filas del dataset
  ord <- order(dataset$numero_de_cliente, dataset$foto_mes)
  pos <- sequence(rle(dataset$numero_de_cliente[ord])$lengths)   # numero de mes dentro de cada cliente
  incompleta <- pos < n                                          # la ventana mezclaria dos clientes
  w <- (1:n) - (n + 1) / 2                                       # pesos de la pendiente de minimos cuadrados
  den <- sum(w^2)
  guardar <- function(nombre, valores) {
    valores[incompleta] <- NA
    res <- rep(NA_real_, length(ord))
    res[ord] <- valores
    set(dataset, j = nombre, value = res)
  }
  for (v in cols_top) {
    x <- as.numeric(dataset[[v]][ord])
    pendiente <- Reduce(`+`, lapply(1:n, function(j) w[j] * shift(x, n - j))) / den
    promedio <- frollmean(x, n)
    guardar(paste0(v, "_tend", n), pendiente)                    # pendiente de los ultimos n meses
    guardar(paste0(v, "_avg", n), promedio)                      # promedio
    guardar(paste0(v, "_max", n), frollapply(x, n, max))         # maximo
    guardar(paste0(v, "_min", n), frollapply(x, n, min))         # minimo
    guardar(paste0(v, "_ratioavg", n), x / promedio)             # valor actual contra su promedio
  }
  message("Bloque tendencias: ", length(cols_top), " variables x 5 = ", 5 * length(cols_top),
          " columnas nuevas (ventana de ", n, " meses)")
}

# ---------- clase binaria, azar y variables del modelo ----------
dataset[, clase01 := ifelse(clase_ternaria %in% PARAM$trainingstrategy$positivos, 1L, 0L)]
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar := runif(nrow(dataset))]

excluir <- c("clase_ternaria", "clase01", "azar", "fold_train", "fold_final_train")
if (PARAM$fe$sacar_ids)   # siguen en el dataset para los lags, los filtros y el archivo de Kaggle
  excluir <- c(excluir, "numero_de_cliente", "foto_mes", grep("^kmes", colnames(dataset), value = TRUE))
campos_buenos <- copy(setdiff(colnames(dataset), excluir))

# los cocientes pueden dar Inf o NaN (division por cero): los paso a NA, que LightGBM maneja bien
no_finitos <- 0
for (j in campos_buenos) if (is.double(dataset[[j]])) {
  malos <- which(is.nan(dataset[[j]]) | is.infinite(dataset[[j]]))
  if (length(malos)) { set(dataset, i = malos, j = j, value = NA_real_); no_finitos <- no_finitos + length(malos) }
}

PARAM$fe$out <- list(ratios_creadas = fe_creadas, ratios_salteadas = fe_salteadas,
                     rankeadas = length(cols_monetarias), tendencias_cols = cols_top,
                     variables_modelo = length(campos_buenos))
message(format(Sys.time(), "%a %b %d %X %Y"), "  preprocesado: ", nrow(dataset), " filas x ",
        length(campos_buenos), " variables del modelo  (", no_finitos, " valores no finitos pasados a NA)")

## Funciones: ganancia, grid search por ganancia, semillerío y Kaggle
Iguales al WF7191.

In [ ]:
# ---------- Kaggle ----------
preparar_kaggle <- function() {
  if (!file.exists("~/.kaggle/kaggle.json"))
    stop("falta ~/.kaggle/kaggle.json en la VM")
  if (Sys.which("kaggle") == "")
    invisible(system("python3 -m pip install -q --user kaggle"))
}

# sube un archivo; si falla (cupo diario, red) lo anota en pendientes_kaggle.txt para reintentar.
# los que ya subieron quedan en subidos_kaggle.txt y no se vuelven a subir al re-correr
# si una subida falla, el resto de la corrida ya no intenta: todo queda pendiente para la proxima vez
kaggle_sin_cupo <- FALSE
subir_a_kaggle <- function(archivo, mensaje) {
  archivo <- normalizePath(archivo)
  if (file.exists(file.path(carpeta_exp, "subidos_kaggle.txt")) &&
      archivo %in% readLines(file.path(carpeta_exp, "subidos_kaggle.txt"))) return(TRUE)
  if (kaggle_sin_cupo) {
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
    return(FALSE)
  }
  salida <- tryCatch(
    suppressWarnings(system2("kaggle", c("competitions", "submit", "-c", PARAM$kaggle$competencia,
                                         "-f", shQuote(archivo), "-m", shQuote(mensaje)),
                             stdout = TRUE, stderr = TRUE)),
    error = function(e) conditionMessage(e))
  ok <- any(grepl("Successfully submitted", salida))
  message("    kaggle ", if (ok) "OK   " else "ERROR", "  ", basename(archivo),
          if (ok) "" else paste0("  ", substr(paste(trimws(salida), collapse = " "), 1, 300)))
  if (ok) {
    cat(archivo, "\n", sep = "", file = file.path(carpeta_exp, "subidos_kaggle.txt"), append = TRUE)
    Sys.sleep(30)
  } else {
    kaggle_sin_cupo <<- TRUE
    cat(archivo, "|", mensaje, "\n", sep = "", file = file.path(carpeta_exp, "pendientes_kaggle.txt"), append = TRUE)
  }
  ok
}

reintentar_pendientes <- function() {
  arch <- file.path(carpeta_exp, "pendientes_kaggle.txt")
  if (!subir_kaggle || !file.exists(arch)) return(invisible())
  pendientes <- unique(setdiff(trimws(readLines(arch)), ""))
  file.remove(arch)
  message(length(pendientes), " archivos pendientes de subir a Kaggle")
  for (p in pendientes) {          # si falla uno, los siguientes quedan anotados sin intentar
    partes <- strsplit(p, "|", fixed = TRUE)[[1]]
    subir_a_kaggle(partes[1], partes[2])
  }
}

# ---------- Training strategy ----------
meses_training <- function(desde) {
  meses <- sort(unique(dataset$foto_mes), decreasing = TRUE)
  meses[meses >= desde & meses <= PARAM$trainingstrategy$hasta]
}

lgb_dataset <- function(filtro) {
  lgb.Dataset(data = data.matrix(dataset[filtro, campos_buenos, with = FALSE]),
              label = dataset[filtro, clase01], free_raw_data = TRUE)
}

# ---------- Ganancia ----------
# ordena por probabilidad, acumula la ganancia de contactar a los primeros N y devuelve el maximo
# de la curva suavizada (promedio movil de PARAM$ganancia$ventana clientes) y el corte donde ocurre
ganancia_maxima <- function(prob, gan) {
  orden <- order(prob, decreasing = TRUE)
  curva <- cumsum(gan[orden])
  suave <- frollmean(curva, PARAM$ganancia$ventana, align = "center")
  corte <- which.max(suave)
  list(ganancia = suave[corte], corte = corte)
}

# ---------- 6.3.2.2 Grid search por GANANCIA en validacion (reanudable) ----------
grid_search <- function(variante, carpeta) {
  archivo <- file.path(carpeta, "tb_grid_search.txt")
  training <- meses_training(variante$desde)
  filtro_train <- dataset[, foto_mes %in% training &
    (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]

  # combinaciones repetidas: un arbol no puede tener mas hojas que filas / min_data_in_leaf, asi que todo
  # num_leaves por encima de ese tope da exactamente el mismo modelo. Se evalua una sola de cada grupo.
  tb <- CJ(num_leaves = PARAM$grid$num_leaves, min_data_in_leaf = PARAM$grid$min_data_in_leaf)
  total <- nrow(tb)
  tb[, hojas_efectivas := pmin(num_leaves, floor(sum(filtro_train) / min_data_in_leaf))]
  tb <- tb[, .(num_leaves = min(num_leaves)), by = .(hojas_efectivas, min_data_in_leaf)][
    order(num_leaves, min_data_in_leaf), .(num_leaves, min_data_in_leaf)]
  message("  grilla: ", total, " combinaciones, ", nrow(tb), " distintas; ",
          PARAM$grid$semillas_eval, " semillas cada una")
  if (file.exists(archivo)) tb <- tb[!fread(archivo), on = .(num_leaves, min_data_in_leaf)]

  if (nrow(tb)) {
    filtro_valid <- dataset[, foto_mes %in% PARAM$trainingstrategy$validate]
    dtrain <- lgb_dataset(filtro_train)
    dvalidate <- lgb_dataset(filtro_valid)
    mvalid <- data.matrix(dataset[filtro_valid, campos_buenos, with = FALSE])
    # ganancia de contactar a cada cliente de validacion, en el mismo orden que dvalidate
    gan_valid <- dataset[filtro_valid, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]

    # metrica para el early stopping: la ganancia maxima suavizada en validacion
    feval_ganancia <- function(preds, dtrain) {
      list(name = "ganancia", value = ganancia_maxima(preds, gan_valid)$ganancia, higher_better = TRUE)
    }

    semillas_eval <- PARAM$semillerio$semillas[seq_len(PARAM$grid$semillas_eval)]
    for (i in seq_len(nrow(tb))) {
      x <- as.list(tb[i])
      evals <- rbindlist(lapply(semillas_eval, function(s) {
        param <- modifyList(PARAM$lgbm$param_fijos, c(x, list(seed = s)))
        modelo <- lgb.train(data = dtrain, valids = list(valid = dvalidate), eval = feval_ganancia,
                            param = param, verbose = -100)
        niter <- modelo$best_iter
        g <- ganancia_maxima(predict(modelo, mvalid, num_iteration = niter), gan_valid)
        rm(modelo); gc(full = TRUE, verbose = FALSE)
        data.table(semilla = s, ganancia = g$ganancia, corte = g$corte, num_iterations = niter)
      }))
      fila <- data.table(tb[i], ganancia = mean(evals$ganancia), ganancia_sd = sd(evals$ganancia),
                         corte = round(mean(evals$corte)), num_iterations = round(mean(evals$num_iterations)))
      fwrite(fila, archivo, sep = "\t", append = file.exists(archivo))
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  nl=", x$num_leaves,
              " mdil=", x$min_data_in_leaf, "  niter ", fila$num_iterations, "  corte ", fila$corte,
              "  ganancia ", round(fila$ganancia / 1e6, 2), " M")
    }
    rm(dtrain, dvalidate, mvalid); gc(full = TRUE, verbose = FALSE)
  }
  res <- fread(archivo)
  setorder(res, -ganancia)
  res
}

# ---------- 6.3.3 Final training con semillerio (reanudable: una prediccion por semilla) ----------
semillerio <- function(variante, carpeta, mejores) {
  carpeta_sem <- file.path(carpeta, "semillas")
  dir.create(carpeta_sem, showWarnings = FALSE)

  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  # modifyList: los mejores del grid PISAN a los fijos (con c() quedaban duplicados y LightGBM usaba los fijos)
  param_final <- modifyList(fijos, mejores)

  pendientes <- PARAM$semillerio$semillas[
    !file.exists(file.path(carpeta_sem, paste0("pred_s", PARAM$semillerio$semillas, ".txt")))]

  final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  if (isTRUE(PARAM$final$escalar_iteraciones)) {
    # el grid eligio las iteraciones con los meses de training; el final_train suma 202107
    factor <- dataset[foto_mes %in% final_train, .N] / dataset[foto_mes %in% meses_training(variante$desde), .N]
    param_final$num_iterations <- round(param_final$num_iterations * factor)
  }

  if (length(pendientes)) {
    dfinal_train <- lgb_dataset(dataset[, foto_mes %in% final_train])
    dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
    mfuture <- data.matrix(dfuture[, campos_buenos, with = FALSE])

    for (s in pendientes) {
      param_final$seed <- s
      modelo <- lgb.train(data = dfinal_train, param = param_final, verbose = -100)
      fwrite(data.table(numero_de_cliente = dfuture$numero_de_cliente, prob = predict(modelo, mfuture)),
             file.path(carpeta_sem, paste0("pred_s", s, ".txt")), sep = "\t")
      if (s == PARAM$semilla_primigenia) {          # modelo e importancia solo de la semilla primigenia
        lgb.save(modelo, file.path(carpeta, "modelo.txt"))
        fwrite(as.data.table(lgb.importance(modelo)), file.path(carpeta, "impo.txt"), sep = "\t")
      }
      message(format(Sys.time(), "%a %b %d %X %Y  "), variante$nombre, "  semilla ", s, " lista")
      rm(modelo); gc(full = TRUE, verbose = FALSE)
    }
    rm(dfinal_train, dfuture, mfuture); gc(full = TRUE, verbose = FALSE)
  }

  # promedio de las probabilidades de todas las semillas
  preds <- rbindlist(lapply(PARAM$semillerio$semillas, function(s)
    fread(file.path(carpeta_sem, paste0("pred_s", s, ".txt")))[, semilla := s]))
  tb_prediccion <- preds[, .(prob = mean(prob)), by = numero_de_cliente]
  fwrite(tb_prediccion, file.path(carpeta, "prediccion.txt"), sep = "\t")   # util para ensembles
  list(prediccion = tb_prediccion, param_final = param_final)
}

# ---------- Archivos para Kaggle ----------
generar_kaggle <- function(variante, carpeta, tb_prediccion) {
  dir.create(file.path(carpeta, "kaggle"), showWarnings = FALSE)
  tb <- copy(tb_prediccion)
  setorder(tb, -prob)
  for (envios in PARAM$kaggle$cortes) {
    tb[, Predicted := 0L]
    tb[1:envios, Predicted := 1L]
    archivo <- file.path(carpeta, "kaggle",
                         paste0("KA", PARAM$experimento, "_", variante$nombre, "_", envios, ".csv"))
    fwrite(tb[, .(numero_de_cliente, Predicted)], archivo, sep = ",")
    if (subir_kaggle)
      subir_a_kaggle(archivo, paste0("z719 ", variante$nombre, " semillerio=", PARAM$semillerio$cantidad,
                                     " semilla=", PARAM$semilla_primigenia, " envios=", envios))
  }
}

## Corridas

In [ ]:
if (subir_kaggle) { preparar_kaggle(); reintentar_pendientes() }
t0 <- Sys.time()
resumen <- list()

for (variante in PARAM$variantes) {
  carpeta <- file.path(carpeta_exp, variante$nombre)
  dir.create(carpeta, showWarnings = FALSE)

  # la reanudacion solo vale si la configuracion es la misma con la que se hicieron los resultados guardados
  firma <- as.yaml(list(semillas_eval = PARAM$grid$semillas_eval, num_leaves = PARAM$grid$num_leaves,
    min_data_in_leaf = PARAM$grid$min_data_in_leaf, param_fijos = PARAM$lgbm$param_fijos,
    ganancia = PARAM$ganancia, training = meses_training(variante$desde),
    validate = PARAM$trainingstrategy$validate, positivos = PARAM$trainingstrategy$positivos,
    training_pct = PARAM$trainingstrategy$training_pct, semillerio = PARAM$semillerio$semillas,
    escalar_iteraciones = PARAM$final$escalar_iteraciones, variables_modelo = campos_buenos))
  archivo_firma <- file.path(carpeta, "config_corrida.yml")
  hay_resultados <- file.exists(file.path(carpeta, "tb_grid_search.txt"))
  if (hay_resultados && !file.exists(archivo_firma))
    stop("La carpeta ", carpeta, " tiene resultados de una version anterior del notebook y no puedo verificar ",
         "con que configuracion se hicieron. Cambia PARAM$experimento para correr en una carpeta nueva.")
  if (hay_resultados && !identical(paste(readLines(archivo_firma), collapse = "\n"), sub("\n$", "", firma)))
    stop("La carpeta ", carpeta, " tiene resultados hechos con OTRA configuracion (semillas del grid, grilla, ",
         "parametros o variables). Reanudar mezclaria resultados. Cambia PARAM$experimento para correr en una ",
         "carpeta nueva, o borra esa carpeta si queres rehacerla.")
  writeLines(sub("\n$", "", firma), archivo_firma)
  message(format(Sys.time(), "%a %b %d %X %Y"), "  ===== ", variante$nombre,
          ": training desde ", variante$desde, " =====")

  # grid search
  tb_grid <- grid_search(variante, carpeta)
  mejores <- as.list(tb_grid[1, .(num_leaves, min_data_in_leaf, num_iterations)])
  message("  mejores: nl=", mejores$num_leaves, " mdil=", mejores$min_data_in_leaf,
          " niter=", mejores$num_iterations, "  ganancia en ", PARAM$trainingstrategy$validate, ": ",
          round(tb_grid[1, ganancia] / 1e6, 2), " M (sd entre semillas ", round(tb_grid[1, ganancia_sd] / 1e6, 2),
          " M) en ", tb_grid[1, corte], " envios")
  print(head(tb_grid, 5))
  if (mejores$num_leaves %in% range(PARAM$grid$num_leaves) ||
      mejores$min_data_in_leaf %in% range(PARAM$grid$min_data_in_leaf))
    warning(variante$nombre, ": el optimo cayo en el borde de la grilla")

  # final training con semillerio y prediccion de 202109
  fin <- semillerio(variante, carpeta, mejores)

  # archivos y subida a Kaggle
  generar_kaggle(variante, carpeta, fin$prediccion)

  # registro de la corrida
  PARAM_var <- PARAM
  PARAM_var$variante <- variante
  PARAM_var$trainingstrategy$training <- meses_training(variante$desde)
  PARAM_var$trainingstrategy$final_train <- c(PARAM$trainingstrategy$validate, meses_training(variante$desde))
  PARAM_var$out$lgbm$ganancia_validacion <- tb_grid[1, ganancia]
  PARAM_var$out$lgbm$corte_validacion <- tb_grid[1, corte]
  PARAM_var$out$lgbm$mejores_hiperparametros <- mejores
  PARAM_var$out$param_final <- fin$param_final
  write_yaml(PARAM_var, file.path(carpeta, "PARAM.yml"))

  resumen[[variante$nombre]] <- data.table(variante = variante$nombre, desde = variante$desde,
    ganancia_valid_M = round(tb_grid[1, ganancia] / 1e6, 2), corte_valid = tb_grid[1, corte], num_leaves = mejores$num_leaves,
    min_data_in_leaf = mejores$min_data_in_leaf, num_iterations = mejores$num_iterations,
    num_iterations_final = fin$param_final$num_iterations, semillas = PARAM$semillerio$cantidad)
}

reintentar_pendientes()
resumen <- rbindlist(resumen)
fwrite(resumen, "resumen_variantes.csv")
message(format(Sys.time(), "%a %b %d %X %Y"), "  terminado en ",
        round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1), " min")
resumen

## Comparación en julio (sin Kaggle)
La mejor combinación del grid de cada corrida, por ganancia en 202107 promediada en 3 semillas. Si la diferencia entre dos corridas es menor que el desvío entre semillas, en la práctica son equivalentes.

In [ ]:
# Comparacion en validacion (202107), sin gastar envios de Kaggle:
# la mejor combinacion del grid de cada corrida que eligio por ganancia (WF7191 y todas las del WF7192)
carpetas <- unlist(lapply(file.path(base, paste0("WF", unique(c(PARAM$comparar_con, PARAM$experimento)))),
                          list.dirs, recursive = FALSE))
comparacion <- rbindlist(lapply(carpetas, function(carp) {
  archivo <- file.path(carp, "tb_grid_search.txt")
  if (!file.exists(archivo)) return(NULL)
  tb <- fread(archivo)
  if (!"ganancia" %in% names(tb)) return(NULL)          # el WF7190 eligio por AUC: no es comparable
  setorder(tb, -ganancia)
  data.table(experimento = basename(dirname(carp)), variante = basename(carp),
             combinaciones = nrow(tb), ganancia_M = round(tb[1, ganancia] / 1e6, 2),
             sd_semillas_M = round(tb[1, ganancia_sd] / 1e6, 2), corte_julio = tb[1, corte],
             num_leaves = tb[1, num_leaves], min_data_in_leaf = tb[1, min_data_in_leaf],
             num_iterations = tb[1, num_iterations])
}))
setorder(comparacion, -ganancia_M)
fwrite(comparacion, "comparacion_validacion.csv")
comparacion

## Puntajes de Kaggle y elección del corte
Curvas del Public del WF7192 junto a las del WF7190 y el WF7191, y corte estable sugerido.

In [ ]:
descargar_envios <- TRUE   # FALSE para reusar el envios_kaggle.csv ya bajado

if (descargar_envios) {
  preparar_kaggle()
  # la CLI lista hasta 200 envios: bajo la lista completa con la API de Python, pagina por pagina
  writeLines(c(
    "import csv, sys",
    "from kaggle.api.kaggle_api_extended import KaggleApi",
    "from kagglesdk.competitions.types.competition_api_service import ApiListSubmissionsRequest",
    "from kagglesdk.competitions.types.competition_enums import SubmissionGroup, SubmissionSortBy",
    "api = KaggleApi()",
    "api.authenticate()",
    "envios, token = [], ''",
    "for pagina in range(50):",
    "    with api.build_kaggle_client() as kaggle:",
    "        req = ApiListSubmissionsRequest()",
    "        req.competition_name = sys.argv[1]",
    "        req.group = SubmissionGroup.SUBMISSION_GROUP_ALL",
    "        req.sort_by = SubmissionSortBy.SUBMISSION_SORT_BY_DATE",
    "        api._set_paging(req, 200, token, -1)",
    "        resp = kaggle.competitions.competition_api_client.list_submissions(req)",
    "    envios.extend(resp.submissions or [])",
    "    token = getattr(resp, 'next_page_token', '') or ''",
    "    if not token or not resp.submissions:",
    "        break",
    "with open(sys.argv[2], 'w', newline='', encoding='utf-8') as f:",
    "    w = csv.writer(f)",
    "    w.writerow(['ref', 'archivo', 'descripcion', 'fecha', 'estado', 'public'])",
    "    for s in envios:",
    "        w.writerow([s.ref, s.file_name, s.description, s.date, s.status, s.public_score])",
    "print(len(envios), 'envios en la competencia')"
  ), "bajar_envios_kaggle.py")
  invisible(system(paste("python3 bajar_envios_kaggle.py", PARAM$kaggle$competencia, "envios_kaggle.csv")))
}

# envios de este notebook y de los experimentos a comparar: KA<experimento>_<variante>_<envios>.csv
patron <- paste0("^KA(", paste(c(PARAM$experimento, PARAM$comparar_con), collapse = "|"), ")_(.+)_([0-9]+)\\.csv$")
envios_kaggle <- fread("envios_kaggle.csv", colClasses = "character")
envios_kaggle[, public := suppressWarnings(as.numeric(public))]
puntajes <- envios_kaggle[grepl(patron, archivo) & grepl("COMPLETE", estado) & !is.na(public)]
puntajes[, variante := paste0(sub(patron, "\\1", archivo), "_", sub(patron, "\\2", archivo))]
puntajes[, envios := as.integer(sub(patron, "\\3", archivo))]
setorder(puntajes, fecha)
puntajes <- puntajes[, .SD[.N], by = .(variante, envios)]   # si se subio dos veces, el mas reciente
fwrite(puntajes[, .(variante, envios, public, fecha, ref, archivo)], "puntajes_kaggle.csv")

# curva de ganancia Public por corte
curva <- dcast(puntajes, envios ~ variante, value.var = "public")
print(curva)

# el pico aislado del Public es ruido: elijo el corte con mejor promedio en una ventana de 3 cortes vecinos
setorder(puntajes, variante, envios)
puntajes[, ventana3 := frollmean(public, 3, align = "center"), by = variante]
mejor <- puntajes[!is.na(ventana3)][order(-ventana3)][1]
fwrite(puntajes, "puntajes_ventanas.csv")
message("Maximo Public: ", puntajes[which.max(public), paste0(variante, " en ", envios, " envios (", public, ")")])
message("Corte estable sugerido: ", mejor$variante, " en ", mejor$envios, " envios",
        "  (promedio de la ventana ", round(mejor$ventana3, 3), ", Public del corte ", mejor$public, ")")
message("Archivo a marcar como entrega final en Kaggle: ", mejor$archivo)

## Curva de ganancia en julio, corte por corte, y comparación con el Public
El grid search guarda solo el máximo de la ganancia en julio. Esta celda guarda **la curva completa**: la ganancia de contactar a los primeros N clientes de 202107, para N de 1 a 3.000.

Entrena la combinación ganadora con los meses de training (sin julio), con la cantidad de árboles que eligió el grid y sin early stopping, igual que se entrena el modelo final. Lo hace con 20 semillas y muestra la curva del **semillerío** (promedio de probabilidades) y la del **modelo individual**. Si ya se bajaron los puntajes de Kaggle, los dibuja encima.

Requiere las celdas 1, 2 y 3 y que la variante ya tenga su grid search. Deja `curva_julio.csv`, `julio_vs_public.csv` y `curva_julio_vs_public.png` en la carpeta de la variante.


In [ ]:
# ===== Curva de ganancia en JULIO (202107), corte por corte, y comparacion con el Public =====
# Requiere haber corrido las celdas 1, 2 y 3, y que la variante ya tenga su grid search.
# Entrena la combinacion ganadora con los meses de training (sin julio), con la cantidad de arboles
# que eligio el grid y SIN early stopping, igual que se entrena el modelo final. Reanudable.

N_SEMILLAS_JULIO <- 20      # cuantas semillas del semillerio usar
MAX_ENVIOS       <- 3000    # hasta donde se dibuja la curva

variante <- PARAM$variantes[[1]]
carpeta  <- file.path(carpeta_exp, variante$nombre)
carpeta_julio <- file.path(carpeta, "julio")
dir.create(carpeta_julio, showWarnings = FALSE)

tb_grid <- fread(file.path(carpeta, "tb_grid_search.txt"))
setorder(tb_grid, -ganancia)
mejores <- as.list(tb_grid[1, .(num_leaves, min_data_in_leaf, num_iterations)])
fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL
param <- modifyList(fijos, mejores)

semillas <- PARAM$semillerio$semillas[seq_len(min(N_SEMILLAS_JULIO, length(PARAM$semillerio$semillas)))]
filtro_valid <- dataset[, foto_mes %in% PARAM$trainingstrategy$validate]
gan_valid   <- dataset[filtro_valid, ifelse(clase_ternaria == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$error)]
baja2_valid <- dataset[filtro_valid, as.integer(clase_ternaria == "BAJA+2")]

# un modelo por semilla, entrenado sin julio, que predice julio
pendientes <- semillas[!file.exists(file.path(carpeta_julio, paste0("pred_s", semillas, ".txt")))]
if (length(pendientes)) {
  filtro_train <- dataset[, foto_mes %in% meses_training(variante$desde) &
    (clase_ternaria %in% PARAM$trainingstrategy$positivos | azar < PARAM$trainingstrategy$training_pct)]
  dtrain <- lgb_dataset(filtro_train)
  mvalid <- data.matrix(dataset[filtro_valid, campos_buenos, with = FALSE])
  for (s in pendientes) {
    param$seed <- s
    modelo <- lgb.train(data = dtrain, param = param, verbose = -100)
    fwrite(data.table(prob = predict(modelo, mvalid)), file.path(carpeta_julio, paste0("pred_s", s, ".txt")))
    message(format(Sys.time(), "%a %b %d %X %Y  "), "julio, semilla ", s, " lista")
    rm(modelo); invisible(gc(full = TRUE, verbose = FALSE))
  }
  rm(dtrain, mvalid); invisible(gc(full = TRUE, verbose = FALSE))
}
probs <- sapply(semillas, function(s) fread(file.path(carpeta_julio, paste0("pred_s", s, ".txt")))$prob)

# curva: ganancia (en millones) de contactar a los primeros N clientes de julio
curva_de <- function(prob) cumsum(gan_valid[order(prob, decreasing = TRUE)])[seq_len(MAX_ENVIOS)] / 1e6
curvas_ind <- apply(probs, 2, curva_de)                      # una columna por semilla
prob_sem   <- rowMeans(probs)                                # el semillerio: promedio de probabilidades
curva <- data.table(
  envios = seq_len(MAX_ENVIOS),
  semillerio = curva_de(prob_sem),
  individual_media = rowMeans(curvas_ind),
  individual_min = apply(curvas_ind, 1, min),
  individual_max = apply(curvas_ind, 1, max),
  baja2_encontrados = cumsum(baja2_valid[order(prob_sem, decreasing = TRUE)])[seq_len(MAX_ENVIOS)]
)
curva[, semillerio_suave := frollmean(semillerio, PARAM$ganancia$ventana, align = "center")]
fwrite(curva, file.path(carpeta, "curva_julio.csv"))

# puntajes Public de esta variante, si ya se bajaron
archivo_puntajes <- file.path(carpeta_exp, "puntajes_kaggle.csv")
nombre_public <- paste0(PARAM$experimento, "_", variante$nombre)
public <- if (file.exists(archivo_puntajes))
  fread(archivo_puntajes)[variante == nombre_public, .(envios, public)] else
  data.table(envios = integer(), public = numeric())

# tabla en los cortes de Kaggle
tabla <- merge(curva[envios %in% PARAM$kaggle$cortes,
                     .(envios, julio_semillerio = round(semillerio, 2), julio_individual = round(individual_media, 2),
                       julio_suave = round(semillerio_suave, 2), baja2_encontrados)],
               public, by = "envios", all.x = TRUE)
tabla[, public_sobre_julio := round(public / julio_semillerio, 2)]
fwrite(tabla, file.path(carpeta, "julio_vs_public.csv"))
print(tabla)

optimo <- curva[which.max(semillerio_suave)]
total_baja2 <- sum(baja2_valid)
message("\nJulio: ", length(gan_valid), " clientes, ", total_baja2, " BAJA+2. Ganancia maxima posible: ",
        round(total_baja2 * PARAM$ganancia$acierto / 1e6, 1), " M")
message("Semillerio de ", length(semillas), ": optimo suavizado en ", optimo$envios, " envios, ",
        round(optimo$semillerio_suave, 2), " M, con ", optimo$baja2_encontrados, " de ", total_baja2, " BAJA+2")
message("Modelo individual promedio en ese corte: ", round(curva[envios == optimo$envios, individual_media], 2),
        " M  (el semillerio le saca ", round(optimo$semillerio - curva[envios == optimo$envios, individual_media], 2), " M)")
meseta <- curva[semillerio_suave >= optimo$semillerio_suave - 1, range(envios)]
message("Meseta (a menos de 1 M del optimo): de ", meseta[1], " a ", meseta[2], " envios")

# grafico: curva de julio y puntos del Public
dibujar <- function() {
  x <- curva[envios >= 300]
  ylim <- range(c(x$individual_min, x$individual_max, x$semillerio, public$public), na.rm = TRUE)
  plot(x$envios, x$semillerio, type = "n", ylim = ylim, xlab = "Envios (clientes contactados)",
       ylab = "Ganancia (millones)", main = paste0("Curva de ganancia: julio vs Public\n", variante$nombre))
  polygon(c(x$envios, rev(x$envios)), c(x$individual_min, rev(x$individual_max)), col = "#D9E2EC", border = NA)
  lines(x$envios, x$individual_media, col = "#7B8794", lwd = 1.5, lty = 2)
  lines(x$envios, x$semillerio, col = "#1F6FB2", lwd = 1)
  lines(x$envios, x$semillerio_suave, col = "#1F6FB2", lwd = 3)
  abline(v = optimo$envios, col = "#1F6FB2", lty = 3)
  if (nrow(public)) points(public$envios, public$public, pch = 19, col = "#C77700", cex = 1.3, type = "b", lwd = 2)
  legend("bottom", bty = "n", cex = 0.9,
         legend = c("Julio: semillerio (suavizado)", "Julio: semillerio (sin suavizar)",
                    "Julio: modelo individual promedio", "Julio: rango entre semillas", "Public de Kaggle (septiembre)"),
         col = c("#1F6FB2", "#1F6FB2", "#7B8794", "#D9E2EC", "#C77700"),
         lwd = c(3, 1, 1.5, 8, 2), lty = c(1, 1, 2, 1, 1), pch = c(NA, NA, NA, NA, 19))
}
png(file.path(carpeta, "curva_julio_vs_public.png"), width = 1400, height = 850, res = 130); dibujar(); invisible(dev.off())
dibujar()